In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

STORAGE_ACCOUNT = "vjretailflow01"
STORAGE_KEY = dbutils.secrets.get(scope="retailflow", key="adls-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print("Setup complete")

In [0]:
products_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/products/")
sellers_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/sellers/")
category_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/category/")

print("\nProducts Schema")
products_bronze.printSchema()
print("\nSellers Schema")
sellers_bronze.printSchema()
print("\nCategory Schema")
category_bronze.printSchema()

In [0]:
def transform_products(products_df, category_df):
    category_clean = category_df \
        .select("product_category_name", "product_category_name_english") \
        .filter(F.col("product_category_name").isNotNull())
    
    return products_df \
        .filter(F.col("product_id").isNotNull()) \
        .withColumn("product_id", F.col("product_id").cast(T.StringType())) \
        .withColumn("product_category_name", F.col("product_category_name").cast(T.StringType())) \
        .withColumn("product_weight_g", F.col("product_weight_g").cast(T.IntegerType())) \
        .withColumn("product_length_cm", F.col("product_length_cm").cast(T.IntegerType())) \
        .withColumn("product_width_cm", F.col("product_width_cm").cast(T.IntegerType())) \
        .withColumn("product_height_cm", F.col("product_height_cm").cast(T.IntegerType())) \
        .withColumn("product_photos_qty", F.col("product_photos_qty").cast(T.IntegerType())) \
        .withColumnRenamed("product_name_lenght", "product_name_length") \
        .withColumnRenamed("product_description_lenght", "product_description_length") \
        .join(category_clean, on="product_category_name", how="left") \
        .withColumn("product_category",
            F.coalesce(
                F.col("product_category_name_english"),
                F.col("product_category_name")
            )
        ) \
        .drop("product_category_name", "product_category_name_english") \
        .withColumn("_products_silver_processed_at", F.current_timestamp()) \
        .drop("_ingested_at", "_ingestion_date", "_source_file")

products_silver = transform_products(products_bronze, category_bronze)

In [0]:
def transform_sellers(df):
    return df \
        .filter(F.col("seller_id").isNotNull()) \
        .withColumn("seller_id", F.col("seller_id").cast(T.StringType())) \
        .withColumn("seller_zip_code_prefix", F.col("seller_zip_code_prefix").cast(T.StringType())) \
        .withColumn("seller_city", F.initcap(F.col("seller_city"))) \
        .withColumn("seller_state", F.upper(F.col("seller_state"))) \
        .withColumn("_silver_processed_at", F.current_timestamp()) \
        .drop("_source_file", "_ingestion_date", "_ingested_at")

sellers_silver = transform_sellers(sellers_bronze)

In [0]:
print(f"Total products: {products_silver.count():,}")
print(f"Null categories: {products_silver.filter(F.col('product_category').isNull()).count():,}")

print("\nTop 10 product categories:")
display(products_silver.groupBy("product_category") \
        .count() \
        .orderBy(F.desc("count")) \
        .limit(10)
)

print("\nTotal Sellers: {sellers_silver.count():,}")
print("\nTop 5 seller states:")
display(sellers_silver.groupBy("seller_state") \
        .count() \
        .orderBy(F.desc("count")) \
        .limit(5)
)

products_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/products/")
sellers_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/sellers/")

print("Data written to silver tables")